# MovieLens 1M：数据读取与基础检查

本 Notebook 当前完成第一步：读取原始数据、展示基本信息并检查数据质量。读取与检查函数位于 `src/data.py`，后续模型可以直接复用。

数据来源：[GroupLens MovieLens 1M](https://grouplens.org/datasets/movielens/1m/)。使用前请阅读数据目录中的 README。

## 1. 导入与定位项目

可从项目根目录或 `notebooks/` 目录启动。请选择安装了项目依赖的 Python 内核，然后按顺序运行全部单元格。

In [1]:
from pathlib import Path
import platform
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / 'src' / 'data.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('请从项目根目录或 notebooks 目录启动 Notebook。')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    column_summary, dataset_summary, load_movielens,
    quality_checks, resolve_data_dir,
)

print(f'Python: {platform.python_version()}')
print(f'pandas: {pd.__version__}')
print(f'Project root: {PROJECT_ROOT}')


Python: 3.14.6
pandas: 3.0.5
Project root: E:\港城莞\课程资料\研一 SemA\CS5481 Data Engineer\data_engineer_group_project


## 2. 读取三张原始表

读取器自动寻找 `data/ml-1m/`、`ml-1m/ml-1m/` 或 `ml-1m/`。如数据位于其他位置，将下面的 `DATA_DIR` 改为对应目录。

文件使用 `::` 分隔，使用 Latin-1 读取电影标题。邮编保持字符串以保留开头的 0；时间戳保持 Unix 秒数；年龄和职业保留官方编码；电影类型暂不拆分。

In [2]:
DATA_DIR = None  # 或指定目录，例如 'ml-1m/ml-1m'
data_dir = resolve_data_dir(DATA_DIR)
tables = load_movielens(data_dir)
users = tables['users']
movies = tables['movies']
ratings = tables['ratings']
print(f'Data directory: {data_dir}')

for name, frame in tables.items():
    print(f'\n{name}: {frame.shape[0]:,} rows, {frame.shape[1]} columns')
    display(frame.head())


Data directory: E:\港城莞\课程资料\研一 SemA\CS5481 Data Engineer\data_engineer_group_project\ml-1m\ml-1m

users: 6,040 rows, 5 columns


,user_id,gender,age,occupation,zip_code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455



movies: 3,883 rows, 3 columns


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy



ratings: 1,000,209 rows, 4 columns


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


## 3. 数据概况汇总

对比三张表的行数、列数、缺失值总数、含缺失值的列和完全重复的记录数。`duplicate_rows` 统计首次出现之后的重复行。实际统计结果以输出为准，不在读取函数中硬编码。

In [3]:
overview = dataset_summary(tables)
display(overview)


,rows,columns,missing_values,columns_with_missing,duplicate_rows
table,,,,,
users,6040,5,0,(none),0
movies,3883,3,0,(none),0
ratings,1000209,4,0,(none),0


## 4. 字段类型、缺失值与唯一值

`unique_values` 不包含缺失值。整数使用 pandas 的可空整数 `Int64`，方便发现缺失值；数值类型并不代表可直接作为连续特征使用，例如 `age` 和 `occupation` 实际是编码。

In [4]:
fields = column_summary(tables)
display(fields)

for name, frame in tables.items():
    print(f'\n{name}')
    frame.info()


dtype  missing_values  unique_values
table   column                                           
users   user_id      Int64               0           6040
        gender      string               0              2
        age          Int64               0              7
        occupation   Int64               0             21
        zip_code    string               0           3439
movies  movie_id     Int64               0           3883
        title       string               0           3883
        genres      string               0            301
ratings user_id      Int64               0           6040
        movie_id     Int64               0           3706
        rating       Int64               0              5
        timestamp    Int64               0         458455


users
<class 'pandas.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     6040 non-null   Int64 
 1   gender      6040 non-null   string
 2   age         6040 non-null   Int64 
 3   occupation  6040 non-null   Int64 
 4   zip_code    6040 non-null   string
dtypes: Int64(3), string(2)
memory usage: 253.8 KB

movies
<class 'pandas.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  3883 non-null   Int64 
 1   title     3883 non-null   string
 2   genres    3883 non-null   string
dtypes: Int64(1), string(2)
memory usage: 94.9 KB

ratings
<class 'pandas.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   user_id    1000209 non-null  

## 5. 评分范围与表间关联检查

检查用户/电影主键重复、用户与电影的重复评分对、超出 1–5 的评分、无法关联的用户/电影 ID，以及负时间戳。缺失字段单独在前面的汇总中报告。电影 ID 不要求连续，也不按最大 ID 推算电影数量。所有检查仅报告结果，不删除或修复记录。

In [5]:
checks = quality_checks(tables)
display(checks)
print(f"Observed rating range: {ratings['rating'].min()} to {ratings['rating'].max()}")

issues = checks.loc[checks['count'] > 0]
if issues.empty:
    print('上述基础质量检查未发现异常；缺失值请结合前面的概况表查看。')
else:
    print('发现需要进一步检查的记录，请根据上表决定后续处理方式。')


,count
check,
duplicate_user_ids,0
duplicate_movie_ids,0
duplicate_user_movie_pairs,0
ratings_outside_1_to_5,0
ratings_with_unknown_user,0
ratings_with_unknown_movie,0
negative_timestamps,0


Observed rating range: 1 to 5
上述基础质量检查未发现异常；缺失值请结合前面的概况表查看。


## 6. 本阶段结果与后续工作

目前已保留原始记录并生成数据预览、表级概况、字段统计和基础质量检查结果。这些输出可用于报告的数据集信息部分。

后续补充评分分布、用户活跃度、电影热度、类型分布及稀疏度分析，再确定清洗规则、标签定义和训练/验证/测试划分。描述性探索可以查看完整数据；模型使用的热度、偏好和相似度等统计特征须在划分后仅使用训练数据计算。

数据引用：F. Maxwell Harper and Joseph A. Konstan. 2015. *The MovieLens Datasets: History and Context.* [DOI: 10.1145/2827872](https://doi.org/10.1145/2827872)。